# Balanced XGBoost hyperparameter comparison

All five runs select 25 features from the training set, use the training class ratio for `scale_pos_weight`, and score the validation set at a 0.5 decision threshold. The test set is reserved for final evaluation.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, average_precision_score, balanced_accuracy_score,
    f1_score, precision_score, recall_score, roc_auc_score,
)
from xgboost import XGBClassifier
import mlflow
import mlflow.sklearn

with np.load("data/processed_datasets.npz") as processed_data:
    X_train = pd.DataFrame(processed_data["X_train"])
    X_val = pd.DataFrame(processed_data["X_val"])
    Y_train = pd.Series(processed_data["Y_train"], name="churn")
    Y_val = pd.Series(processed_data["Y_val"], name="churn")

N_FEATURES = 25
THRESHOLD = 0.5
assert X_train.shape[1] >= N_FEATURES
positive_class_weight = (Y_train == 0).sum() / (Y_train == 1).sum()
print("Training:", X_train.shape, "Validation:", X_val.shape)
print("XGBoost positive class weight:", positive_class_weight)

mlflow.set_tracking_uri("sqlite:///mlflow_churn.db")
mlflow.set_experiment("churn-xgboost-balanced-hyperparameters")

In [ ]:
param_configs = [
    {
        "learning_rate": 0.03,
        "max_depth": 3,
        "n_estimators": 400,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
    },
    {
        "learning_rate": 0.05,
        "max_depth": 3,
        "n_estimators": 400,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
    },
    {
        "learning_rate": 0.05,
        "max_depth": 5,
        "n_estimators": 400,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
    },
    {
        "learning_rate": 0.1,
        "max_depth": 5,
        "n_estimators": 200,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
    },
    {
        "learning_rate": 0.1,
        "max_depth": 7,
        "n_estimators": 200,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
    },
]

In [ ]:
def evaluate_probabilities(y, probabilities, threshold):
    predictions = (probabilities >= threshold).astype(int)
    return {
        "accuracy": accuracy_score(y, predictions),
        "balanced_accuracy": balanced_accuracy_score(y, predictions),
        "precision": precision_score(y, predictions, zero_division=0),
        "recall": recall_score(y, predictions, zero_division=0),
        "f1": f1_score(y, predictions, zero_division=0),
        "roc_auc": roc_auc_score(y, probabilities),
        "average_precision": average_precision_score(y, probabilities),
    }


results = []
for config_number, params in enumerate(param_configs, start=1):
    run_name = f"balanced_xgboost_top_{N_FEATURES}_config_{config_number}"
    print("Running:", run_name, params, flush=True)
    pipeline = Pipeline([
        ("feature_selection", SelectKBest(score_func=f_classif, k=N_FEATURES)),
        ("model", XGBClassifier(
            **params,
            scale_pos_weight=positive_class_weight,
            random_state=42,
            eval_metric="logloss",
            n_jobs=-1,
        )),
    ])

    with mlflow.start_run(run_name=run_name):
        pipeline.fit(X_train, Y_train)
        probabilities = pipeline.predict_proba(X_val)[:, 1]
        metrics = evaluate_probabilities(Y_val, probabilities, THRESHOLD)
        mlflow.log_params({
            "model_type": "xgboost",
            "balance": "balanced",
            "n_features": N_FEATURES,
            "threshold": THRESHOLD,
            "evaluation_split": "validation",
            "scale_pos_weight": positive_class_weight,
            **params,
        })
        mlflow.log_metrics(metrics)
        mlflow.sklearn.log_model(
            pipeline, name="model", serialization_format="cloudpickle"
        )

    results.append({
        "config": config_number,
        **params,
        "n_features": N_FEATURES,
        "threshold": THRESHOLD,
        **metrics,
    })
    print("Finished:", run_name, "F1:", round(metrics["f1"], 4), flush=True)

In [ ]:
results_df = pd.DataFrame(results).sort_values(
    by="f1", ascending=False
).reset_index(drop=True)
results_df.to_csv("data/xgboost_hyperparameter_results.csv", index=False)
results_df